In [1]:
pip install -U langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-huggingface langchain-ollama langchain-openrouter openai tiktoken rapidocr-onnxruntime python-dotenv sentence-transformers pypdf

  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
Using cached langchain_ollama-1.1.0-py3-none-any.whl (31 kB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)

   ------------- -------------------------- 1/3 [langchain-openai]
   ---------------------------------------- 3/3 [langchain-ollama]

Note: you may need to restart the kernel to use updated packages.


In [2]:
import dotenv
dotenv.load_dotenv(override=True)

True

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS

c:\Users\LOQ\anaconda3\envs\Rag_Env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
loader=TextLoader('state_of_the_union.txt', encoding="utf8")

In [9]:
document=loader.load()

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

In [18]:
text_chunks=text_splitter.split_documents(document)

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2181.53it/s]


In [20]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [29]:
retriever=vectorstore.as_retriever()

In [22]:
from langchain_core.prompts import ChatPromptTemplate

In [23]:
template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [24]:
prompt=ChatPromptTemplate.from_template(template)

In [26]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [27]:
output_parser=StrOutputParser()

In [35]:
llm = ChatOpenRouter(
    model="poolside/laguna-m.1:free",
    temperature=0
)

In [36]:
rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [42]:
rag_chain.invoke("How is the United States supporting Ukraine economically and militarily?")

'\nThe United States is providing over $1 billion in direct economic assistance to Ukraine, as stated in the context. Militarily, the U.S. is not engaging directly in Ukraine but has mobilized ground forces, air squadrons, and naval deployments to defend NATO allies in Europe, including Poland, Romania, and the Baltic states. This support aims to deter further Russian aggression and protect NATO territory, with the U.S. emphasizing its commitment to defending every inch of allied land. The assistance is part of a broader strategy to uphold stability in Europe and counter Putin’s unprovoked invasion. The U.S. also highlights its diplomatic and collective resolve with allies to ensure costs for aggression are imposed. Humanitarian aid is mentioned alongside economic and military support, though specific details are not provided. The context underscores that U.S. actions are coordinated with NATO and other partners. It clarifies that American forces are not in Ukraine but are positioned t